# CardioSafe AI - Notebook 11: collect every model and metric in one place
Reads the result files saved by notebooks 02 to 09 from your Drive and prints one report: which models exist, ECG metrics, calibration, uncertainty, conformal coverage, QTc agreement, OCR, scenario checks. Nothing is retrained; it takes under a minute (CPU is fine).
It also saves `FINAL_METRICS.md` and `FINAL_METRICS.xlsx` in your CardioSafe folder for the report.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
WORK = '/content/drive/MyDrive/CardioSafe'
def show(title, fn, index_col=0, round_=3):
    p = os.path.join(WORK, fn)
    if not os.path.exists(p):
        print(f'\n### {title}\n(missing: {fn} - run the notebook that creates it)'); return None
    d = pd.read_csv(p, index_col=index_col)
    print(f'\n### {title}  [{fn}]'); print(d.round(round_).to_string()); return d
print('files found:', sorted(os.path.basename(f) for f in glob.glob(WORK + '/results_*.csv')))

Mounted at /content/drive
files found: ['results_02_baselines.csv', 'results_03_final_comparison.csv', 'results_03_runs.csv', 'results_04_calibration.csv', 'results_04_conformal.csv', 'results_04_uncertainty_auroc.csv', 'results_04_uncertainty_error.csv', 'results_05_faithfulness.csv', 'results_05_faithfulness_per_class.csv', 'results_06_qt_agreement.csv', 'results_06_qtc_flag.csv', 'results_06b_agreement.csv', 'results_06b_benchmarks.csv', 'results_06b_flag.csv', 'results_06b_three_way.csv', 'results_08_ocr.csv', 'results_09_scenarios.csv']


## 1. Which models exist

In [2]:
ck = sorted(os.path.basename(f) for f in glob.glob(WORK + '/models/*.pt')) + sorted(os.path.basename(f) for f in glob.glob(WORK + '/cnn1d_s*.pt'))
print(len(ck), 'saved neural-network checkpoints:'); print(ck)
t = json.load(open(WORK + '/trust_config.json')); print('\nECG ensemble used by the demo (%d members):' % len(t['members']), t['members'])
q = json.load(open(WORK + '/qtc_config_learned.json')); print('QT/RR estimator checkpoints:', q['checkpoints'])
print('\nOCR engines compared: tesseract_psm6, tesseract_psm4, easyocr | drug resolver: brand table + RxNorm (rule-based, not trained)')

23 saved neural-network checkpoints:
['bigru_s0.pt', 'bigru_s1.pt', 'bigru_s2.pt', 'cnn_transformer_s0.pt', 'cnn_transformer_s1.pt', 'cnn_transformer_s2.pt', 'inception1d_s0.pt', 'inception1d_s1.pt', 'inception1d_s2.pt', 'qtnet_s0.pt', 'qtnet_s1.pt', 'qtnet_s2.pt', 'resnet1d_focal_s0.pt', 'resnet1d_noaug_s0.pt', 'resnet1d_s0.pt', 'resnet1d_s1.pt', 'resnet1d_s2.pt', 'xresnet1d_s0.pt', 'xresnet1d_s1.pt', 'xresnet1d_s2.pt', 'cnn1d_s0.pt', 'cnn1d_s1.pt', 'cnn1d_s2.pt']

ECG ensemble used by the demo (15 members): ['resnet1d_s0', 'resnet1d_s1', 'resnet1d_s2', 'xresnet1d_s0', 'xresnet1d_s1', 'xresnet1d_s2', 'inception1d_s0', 'inception1d_s1', 'inception1d_s2', 'bigru_s0', 'bigru_s1', 'bigru_s2', 'cnn_transformer_s0', 'cnn_transformer_s1', 'cnn_transformer_s2']
QT/RR estimator checkpoints: ['models/qtnet_s0.pt', 'models/qtnet_s1.pt', 'models/qtnet_s2.pt']

OCR engines compared: tesseract_psm6, tesseract_psm4, easyocr | drug resolver: brand table + RxNorm (rule-based, not trained)


## 2. ECG classification (test fold 10; thresholds tuned on validation fold 9)

In [3]:
base = show('Classical / first deep baselines', 'results_02_baselines.csv')
runs = show('Every deep run (architecture x seed)', 'results_03_runs.csv')
fin = show('Final comparison of all models', 'results_03_final_comparison.csv')


### Classical / first deep baselines  [results_02_baselines.csv]
                  AUROC  AUPRC     F1  AUROC_lo  AUROC_hi  AUROC_NORM  AUROC_MI  AUROC_STTC  AUROC_CD  AUROC_HYP  F1_NORM  F1_MI  F1_STTC  F1_CD  F1_HYP  train_s
model                                                                                                                                                            
logreg            0.865  0.692  0.654     0.856     0.874       0.889     0.832       0.875     0.854      0.876    0.794  0.618    0.642  0.663   0.552     12.0
decision_tree     0.750  0.507  0.538     0.737     0.765       0.782     0.699       0.745     0.757      0.768    0.694  0.503    0.507  0.558   0.428     13.0
random_forest     0.865  0.706  0.655     0.856     0.874       0.893     0.827       0.881     0.839      0.882    0.796  0.610    0.648  0.629   0.593    206.0
xgboost           0.890  0.762  0.688     0.882     0.899       0.913     0.859       0.912     0.875      0.891    0.820  0

## 3. Trustworthiness: calibration, uncertainty, conformal prediction

In [4]:
cal = show('Calibration (ECE / Brier / NLL before and after temperature scaling)', 'results_04_calibration.csv')
show('Uncertainty: error rate vs coverage', 'results_04_uncertainty_error.csv')
show('Uncertainty: AUROC of uncertainty scores for detecting errors', 'results_04_uncertainty_auroc.csv')
con = show('Conformal prediction (target coverage 90%)', 'results_04_conformal.csv')


### Calibration (ECE / Brier / NLL before and after temperature scaling)  [results_04_calibration.csv]
                          T  ECE_before  ECE_after  Brier_before  Brier_after  NLL_before  NLL_after  AUROC_before  AUROC_after
model                                                                                                                          
cnn1d_s0 (no aug)     1.156       0.043      0.043         0.087        0.087       0.285      0.282         0.927        0.927
resnet1d_s0           1.179       0.038      0.035         0.086        0.085       0.284      0.281         0.922        0.922
resnet1d_focal_s0     0.465       0.112      0.022         0.096        0.082       0.332      0.270         0.925        0.925
xresnet1d_seedavg     1.111       0.032      0.031         0.080        0.080       0.265      0.263         0.932        0.932
ENSEMBLE (15 models)  1.013       0.026      0.025         0.078        0.078       0.257      0.257         0.934        0.934


## 4. QTc estimator (learned) vs benchmarks

In [5]:
show('Benchmarks', 'results_06b_benchmarks.csv'); show('Agreement with reference QT / QTc', 'results_06b_agreement.csv')
show('Prolonged-QTc flag accuracy', 'results_06b_flag.csv'); show('Three-way normal / borderline / prolonged', 'results_06b_three_way.csv')


### Benchmarks  [results_06b_benchmarks.csv]
                                         comparison     n  bias_ms   MAE_ms  SD_diff_ms  LoA_low  LoA_high  pearson_r  within_20ms  within_30ms  within_40ms
quantity                                                                                                                                                    
QT              Uni-G vs 12SL (two real algorithms)  2155    0.134    8.728      16.923  -33.036    33.304      0.904        0.932        0.963        0.976
QT                       constant predictor vs 12SL  2155    0.550   29.104      38.215  -74.350    75.451     -0.000        0.448        0.613        0.742
RR              Uni-G vs 12SL (two real algorithms)  2158    0.956    2.364      11.551  -21.685    23.597      0.998        0.977        0.981        0.987
RR                       constant predictor vs 12SL  2158    0.369  138.073     176.519 -345.609   346.347      0.000        0.098        0.148        0.195
QTc Frideric

,n,share_%,12SL says prolonged_%
outcome,,,
normal,1459,67.608897,0.616861
borderline,663,30.722892,23.076923
prolonged,36,1.668211,97.222222


## 5. Prescription pipeline and combined module

In [6]:
ocr = show('OCR + drug extraction (synthetic printed prescriptions)', 'results_08_ocr.csv', index_col=[0, 1])
sc = show('Combined-module scenarios', 'results_09_scenarios.csv', index_col=0)


### OCR + drug extraction (synthetic printed prescriptions)  [results_08_ocr.csv]
                       n_prescriptions  sec_per_image  precision  recall     f1   TP  FP  FN  unrecognised_lines
engine         level                                                                                            
tesseract_psm6 easy                 40          1.149        1.0   0.993  0.996  141   0   1                   1
               medium               40          1.332        1.0   0.974  0.987  151   0   4                   3
               hard                 40          1.369        1.0   0.879  0.935  123   0  17                   9
tesseract_psm4 easy                 40          0.969        1.0   0.993  0.996  141   0   1                   1
               medium               40          1.231        1.0   0.974  0.987  151   0   4                   3
               hard                 40          1.254        1.0   0.857  0.923  120   0  20                  10
easyocr      

In [7]:
# one-page markdown + Excel for the report
parts = []
def add(t, d):
    if d is not None: parts.append(f'## {t}\n\n' + d.round(3).to_markdown() + '\n')
add('ECG classification: final comparison', fin); add('ECG classification: baselines', base); add('Calibration', cal); add('Conformal prediction', con); add('OCR and drug extraction', ocr); add('Combined-module scenarios', sc)
try:
    open(WORK + '/FINAL_METRICS.md', 'w').write('# CardioSafe AI - final metrics\n\n' + '\n'.join(parts))
except Exception as e:
    print('markdown export skipped:', e)
with pd.ExcelWriter(WORK + '/FINAL_METRICS.xlsx') as xw:
    for n, d in [('ecg_final', fin), ('ecg_baselines', base), ('ecg_runs', runs), ('calibration', cal), ('conformal', con), ('ocr', ocr), ('scenarios', sc)]:
        if d is not None: d.to_excel(xw, sheet_name=n)
print('saved FINAL_METRICS.md and FINAL_METRICS.xlsx in', WORK)

saved FINAL_METRICS.md and FINAL_METRICS.xlsx in /content/drive/MyDrive/CardioSafe


When it finishes, send me the printed text of sections 2, 3 and 4, or upload the executed notebook, and I will write the results summary for your report.